# Image Classification on CIFAR-10 via ResNet
The goal of this task is to perform image classification on the [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) dataset using the [ResNet](https://arxiv.org/pdf/1512.03385.pdf) architecture. CIFAR-10 is a dataset consisting of 60,000 32x32 color images in 10 classes (airplanes, cars, birds, cats, deer, dogs, frogs, horses, ships, and trucks), with 6,000 images per class. The ResNet (Residual Network) is a deep learning architecture known for its ability to train very deep neural networks effectively. A typical deep learning training paradigm includes:

1. data preparation
2. neural network definition
3. loss function definition
4. training
5. evaluation

To better prepare for this test, you are recommended to read Section 3.2-3.3 and Figure 2, Table 1 which illustrates the architecture of ResNet-18 from the ResNet paper [1].

[1] Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun. "Deep Residual Learning for Image Recognition." In Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR). 2016.

## Before Starting
Before your test begins, we offer a comprehensive runnable example to help you become familiar with standard deep learning training techniques for computer vision tasks.

### Install necessary packages

In [ ]:
!pip3 install torch torchvision

#### Import necessary package

In [ ]:
import torch
import torchvision
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import torchvision.transforms as transforms

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
print(device)

#### Loading (downloading) the dataset

In [ ]:
batch_size = 4
train_transforms = torchvision.transforms.PILToTensor()
trainset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=train_transforms
)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=batch_size, shuffle=True, num_workers=2
)

testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True,transform=train_transforms
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=32, shuffle=False, num_workers=2
)

We show some examples in the Cifar10.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# functions to show an image

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
def imshow(img):
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


# get some random training images
dataiter = iter(testloader)
images, labels = next(dataiter)

# show images
imshow(torchvision.utils.make_grid(images, nrow=8))
# print labels
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(32)))

### Define a very simple network
The network is constructed via multiple fully connected layer called `nn.Linear()` and activation function named `nn.ReLU()`.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(3*32*32, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


In [ ]:
net = Net().to(device)

### Define a loss function

In [ ]:
def loss_function(outputs, labels):
    batch_size = outputs.size(0)
    # Calculate softmax probabilities
    softmax_outputs = F.softmax(outputs, dim=1)
    # Extract the probabilities corresponding to the true class labels
    true_class_probs = softmax_outputs[range(batch_size), labels]
    # Calculate the negative log likelihood loss
    loss = -torch.log(true_class_probs).mean()
    return loss
criterion = loss_function
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

### Training loop

In [ ]:
for epoch in range(2):
    running_loss = 0.0
    for i, data in enumerate(tqdm(trainloader)):
        inputs, labels = data
        inputs = inputs.to(device) / 255.0
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:  # print every 2000 mini-batches
            print(f"epoch: {epoch + 1}, step: {i + 1:5d}, loss: {running_loss / 2000:.3f}")
            running_loss = 0.0

print("Finished Training")

### Evaluation loop

In [ ]:
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images = images / 255.0 * 2 - 1
        images = images.to(device)
        labels = labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of the network on the 10000 test images: {100 * correct // total} %")

## Q1 Using ResNet-18
The above network achieves ~35% accuracy. In this section, we are going to improve the performance via adding data augmentation, changing the simple network into ResNet-18. You are required to fill missing parts in the three following parts in Q1:
1. Data augmentation
2. Network definition
3. Loss function

### Q1.1 Utilize data augmentation
The adoption of data augmentation not only improves the model's robustness to variations in the input data but also contributes to better generalization performance. You are required to fill **2 lines** using pytorch pre-defined functions:
1. random crop the image into size 32*32 with 4 as the padding.
2. random horizontal flip.

In [ ]:
transform_train = transforms.Compose([
    # Q1.1 line 1
    # Q1.1 line 2
    transforms.ToTensor(),
    transforms.Normalize(0.5, 0.5)
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(0.5, 0.5)
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

### Q 1.2 Complete ResNet-18
ResNet-18 is constructed by a stack of layers which contains a convolution layer, a batch normalization layer and, an activate function. Layers are connected via residual connection.
You are required to fill **3 lines** of the input layer in the network using pytorch pre-defined functions:
1. a convolution function with input channel 3, output channel 64, kernel size 3, stride 1 and padding size 1.
2. a batch normalization function with input channel 64.
3. a relu activation function.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, inchannel, outchannel, stride=1):
        super(ResidualBlock, self).__init__()
        self.left = nn.Sequential(
            nn.Conv2d(
                inchannel,
                outchannel,
                kernel_size=3,
                stride=stride,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(outchannel),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                outchannel, outchannel, kernel_size=3, stride=1, padding=1, bias=False
            ),
            nn.BatchNorm2d(outchannel),
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or inchannel != outchannel:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    inchannel, outchannel, kernel_size=1, stride=stride, bias=False
                ),
                nn.BatchNorm2d(outchannel),
            )

    def forward(self, x):
        out = self.left(x)
        out = out + self.shortcut(x)
        out = F.relu(out)

        return out


class ResNet(nn.Module):
    def __init__(self, ResidualBlock, num_classes=10):
        super(ResNet, self).__init__()
        self.inchannel = 64
        # Q1.2
        self.conv1 = nn.Sequential(
            # Q1.2 line 1
            # Q1.2 line 2
            # Q1.2 line 3
        )
        self.layer1 = self.make_layer(ResidualBlock, 64, 2, stride=1)
        self.layer2 = self.make_layer(ResidualBlock, 128, 2, stride=2)
        self.layer3 = self.make_layer(ResidualBlock, 256, 2, stride=2)
        self.layer4 = self.make_layer(ResidualBlock, 512, 2, stride=2)
        self.fc = nn.Linear(512, num_classes)

    def make_layer(self, block, channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.inchannel, channels, stride))
            self.inchannel = channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

In [ ]:
net = ResNet(ResidualBlock).to(device)

### Q 1.3 Complete loss function
The cross entropy loss is the most comman chiose when training on multi-class classification task. You are required to fill **1 line** using pytorch pre-defined function:
1. a cross entropy loss function

In [ ]:
criterion = # Q1.3 line 1
optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

Reuse the previous training and evaluation loop

In [ ]:
for epoch in range(10):
    running_loss = 0.0
    for i, data in enumerate(tqdm(trainloader)):
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 100 == 99:
            print(f"epoch: {epoch + 1}, step: {i + 1:5d}, loss: {running_loss / 100:.3f}")
            running_loss = 0.0

print("Finished Training")

In [ ]:
correct = 0
total = 0
with torch.no_grad():
    for data in tqdm(testloader):
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of the network on the 10000 test images: {100 * correct // total} %")

## Q2 Further improving
We have achieved a much better performance using ResNet-18. In Q2, you are required to further improve the performance of the classification accuracy considering the following three possible directions:
1. Deeper ResNet
2. Hyper-parameter tuning
3. Drop-out

### Q2.1 Deeper ResNet
The previous used ResNet-18 means that the network is constructed via 18 layers. It is well know that the network could have better performance in most cases when we make it deeper in **deep** learning. You could explore using ResNet-34, ResNet-50.

Ref:
1. https://pytorch.org/vision/main/models/resnet.html
2. https://arxiv.org/pdf/1512.03385.pdf


### Q2.2 Hyper-parameter Tuning
Different hyper-parameters could lead to significant different performance of teh same model. You could explore different combinations of learning rate, batch size, and number of training epochs.

### Q2.3 Drop out
As we are facing a relative-small dataset, it is possible that the network only remenbers but not learns from the training set which may hurt the performance on the test set. You could explore adding a drop out function at the end of the ResNet.

Ref:
1. https://www.cs.toronto.edu/~rsalakhu/papers/srivastava14a.pdf
2. https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html